# Lab 3.3, Build 3: Pack the context

Filtered retrieval gives Tina the right documents. The context window decides how
many of them she reads. Two result-set shapes, two strategies, two seeded budgets.

Run the harness cell, then complete the two cells marked **YOUR WORK**.


In [ ]:
# Harness. Nothing here is graded.
import json
import os
import pathlib
import sys

sys.path.insert(0, "/opt/ara/lib")

from elasticsearch import Elasticsearch
from ara_metrics import context_fit, token_count

ES = Elasticsearch(os.environ["ES_URL"], api_key=os.environ["ES_API_KEY"], request_timeout=120)
TRACES = pathlib.Path("/home/elastic/.traces")
TRACES.mkdir(exist_ok=True)

BUDGETS = json.loads(pathlib.Path("/home/elastic/constraint.json").read_text())
SMALL = int(BUDGETS["context_budget_small"])
LARGE = int(BUDGETS["context_budget_large"])

DEV_A = json.loads(pathlib.Path("/home/elastic/dev-sets/dev-queries.json").read_text())[:4]
DEV_B = json.loads(pathlib.Path("/home/elastic/dev-sets/dev-queries-passages.json").read_text())[:4]

SET_B_BUCKETS = ["bucket_20k", "bucket_40k"]


def candidates(set_name: str, query_text: str) -> list:
    """Retrieve the candidate list for a set. The grader retrieves the same way."""
    if set_name == "A":
        index = "cortex-cases"
        body = {
            "retriever": {"rrf": {"retrievers": [
                {"standard": {"query": {"match": {"body_text": query_text}}}},
                {"standard": {"query": {"semantic": {"field": "body", "query": query_text}}}},
            ], "rank_window_size": 50, "rank_constant": 60}},
            "size": 12,
        }
    else:
        index = "cortex-sar-passages"
        body = {
            "retriever": {"standard": {"query": {"bool": {
                "must": [{"semantic": {"field": "body", "query": query_text}}],
                "filter": [{"terms": {"bucket": SET_B_BUCKETS}}],
            }}}},
            "size": 3,
        }
    response = ES.search(index=index, body=body)
    out = []
    for hit in response["hits"]["hits"]:
        source = hit.get("_source", {})
        text = source.get("body_text") or source.get("body") or ""
        if isinstance(text, dict):
            text = text.get("text", "")
        out.append({"doc_id": source.get("doc_id") or hit["_id"], "text": text, "body": text})
    return out


print(f"budgets for this sandbox: small {SMALL}, large {LARGE}")
sample = candidates("A", DEV_A[0]["query_text"])
print(f"set A returns {len(sample)} memos, "
      f"{[token_count(item['text']) for item in sample[:5]]} tokens each")
sample = candidates("B", DEV_B[0]["query_text"])
print(f"set B returns {len(sample)} sections, "
      f"{[token_count(item['text']) for item in sample]} tokens each")


## Strategy 1: rerank, then take whole passages

Score every candidate against the query, then fill the budget in that order.


In [ ]:
# YOUR WORK: pack by reranking, then taking whole passages while they fit.
import os

from ara_metrics import token_count
from ara_pack import rerank_top_n

MAX_CANDIDATES = 30  # cap before reranking: a long candidate list is slow


def pack_rerank_top_n(results, query: str, budget: int) -> str:
    """Score every candidate, then take whole passages in that order until full."""
    pool = list(results)[:MAX_CANDIDATES]
    if not pool:
        return ""
    ranked = rerank_top_n(pool, query, n=len(pool), rerank_id=os.environ["ARA_RERANK_ID"])
    parts, used = [], 0
    for item in ranked:
        text = item.get("text") or item.get("body") or ""
        # TODO: measure this passage, decide whether it fits, and decide what to do
        # when it does not. Skipping and scanning on is a different strategy from
        # stopping, and the difference shows up on set B.
    return "\n\n".join(parts)


## Strategy 2: compress each candidate first

Summarize every candidate to a per-document budget, then fill the budget with the
summaries. The per-document budget is the whole design of this strategy.


In [ ]:
# YOUR WORK: pack by compressing each candidate first, then taking the summaries.
import os

from ara_metrics import token_count
from ara_pack import summarize_first


def pack_summarize_first(results, query: str, budget: int) -> str:
    """Compress each candidate to a per-document budget, then pack the summaries."""
    pool = list(results)[:MAX_CANDIDATES]
    if not pool:
        return ""
    # TODO: choose the per-document budget. Divide the budget by the number of
    # candidates and every document fits, but the relevant span may not survive.
    # Leave head room: a summary can come back longer than the number you ask for.
    per_doc = 0
    summarized = summarize_first(pool, query,
                                 os.environ["ARA_INFERENCE_COMPLETION_ID"], per_doc)
    parts, used = [], 0
    for item in summarized:
        text = item.get("body") or item.get("text") or ""
        # TODO: pack the summaries the same way you packed whole passages.
    return "\n\n".join(parts)


# Provided: the entry point the harness and the grader call.
def pack_context(results, query: str, budget: int, strategy: str) -> str:
    if strategy == "rerank_top_n":
        return pack_rerank_top_n(results, query, budget)
    if strategy == "summarize_first":
        return pack_summarize_first(results, query, budget)
    raise ValueError(f"Unknown strategy: {strategy}")


In [ ]:
# Run the eight combinations: two sets, two strategies, two budgets.
TABLE, SAMPLES = [], []

for set_name, dev_set in (("A", DEV_A), ("B", DEV_B)):
    pool_by_query = {q["query_id"]: candidates(set_name, q["query_text"]) for q in dev_set}
    for strategy in ("rerank_top_n", "summarize_first"):
        for budget in (SMALL, LARGE):
            measured, retained = [], 0
            first_packed, first_id = "", dev_set[0]["query_id"]
            for position, query in enumerate(dev_set):
                packed = pack_context(pool_by_query[query["query_id"]],
                                      query["query_text"], budget, strategy)
                if position == 0:
                    first_packed = packed
                measured.append(token_count(packed))
                if query["gold_literal"] in packed:
                    retained += 1
            retention = retained / len(dev_set)
            TABLE.append({
                "set": set_name,
                "strategy": strategy,
                "budget": budget,
                "packed_tokens": max(measured) if measured else 0,
                "fit": all(count <= budget for count in measured),
                "gold_retention": round(retention, 3),
                "answer_accuracy": round(retention, 3),
            })
            SAMPLES.append({
                "set": set_name,
                "strategy": strategy,
                "budget": budget,
                "query_id": first_id,
                "packed_context": first_packed,
            })
            print(f"set {set_name}  {strategy:<16} budget {budget:>6}  "
                  f"max {max(measured) if measured else 0:>6} tokens  "
                  f"retention {retention:.2f}")

print("\nRead the two set B rows at the smaller budget. The Defend asks about them.")
print(f"context_fit on the last sample: {context_fit(SAMPLES[-1]['packed_context'], SAMPLES[-1]['budget'])}")


In [ ]:
# Save the results file the grader reads.
payload = {
    "budgets": {"small": SMALL, "large": LARGE},
    "table": TABLE,
    "samples": SAMPLES,
}
(TRACES / "pack-results.json").write_text(json.dumps(payload, indent=2))
print(f"pack-results.json written with {len(TABLE)} rows and {len(SAMPLES)} samples.")
print("Select Check.")
